# Equal-Weight S&P 500 Index Fund

## Introduction & Library Imports

The S&P 500 is the world's most popular stock market index. The largest fund that is benchmarked to this index is the SPDR® S&P 500® ETF Trust. It has more than US$250 billion of assets under management.

The goal of this section of the course is to create a Python script that will accept the value of your portfolio and tell you how many shares of each S&P 500 constituent you should purchase to get an equal-weight version of the index fund.

## Library Imports

The first thing we need to do is import the open-source software libraries that we'll be using in this tutorial.

In [17]:
import numpy as np
import pandas as pd
import requests
import xlsxwriter
import math

## Importing Our List of Stocks

The next thing we need to do is import the constituents of the S&P 500.

These constituents change over time, so in an ideal world you would connect directly to the index provider (Standard & Poor's) and pull their real-time constituents on a regular basis.

Paying for access to the index provider's API is outside of the scope of this course. 

There's a static version of the S&P 500 constituents available here. [Click this link to download them now](https://drive.google.com/file/d/1ZJSpbY69DVckVZlO9cC6KkgfSufybcHN/view?usp=sharing). Move this file into the `starter-files` folder so it can be accessed by other files in that directory.

Now it's time to import these stocks to our Jupyter Notebook file.

In [18]:
stocks = pd.read_csv('data/sp_500_stocks.csv')
stocks.head()

,Ticker
0,A
1,AAL
2,AAP
3,AAPL
4,ABBV


## Acquiring an API Token

Now it's time to import our IEX Cloud API token. This is the data provider that we will be using throughout this course.

API tokens (and other sensitive information) should be stored in a `secrets.py` file that doesn't get pushed to your local Git repository. We'll be using a sandbox API token in this course, which means that the data we'll use is randomly-generated and (more importantly) has no cost associated with it.

[Click here](http://nickmccullum.com/algorithmic-trading-python/secrets.py) to download your `secrets.py` file. Move the file into the same directory as this Jupyter Notebook before proceeding.

In [19]:
api_token = 'pk_2232a975d7f0498bb545463065fb960b' 

## Making Our First API Call

Now it's time to structure our API calls to IEX cloud. 

We need the following information from the API:

* Market capitalization for each stock
* Price of each stock



In [24]:
symbol = 'AAPL'
api_url = f'https://api.iex.cloud/v1/data/core/quote/{symbol}?token={api_token}'
print(api_url)
data = requests.get(api_url).json()[0]
print(data)

https://api.iex.cloud/v1/data/core/quote/AAPL?token=pk_2232a975d7f0498bb545463065fb960b
{'avgTotalVolume': 68005647, 'calculationPrice': 'close', 'change': 0.52, 'changePercent': 0.00305, 'close': 171.21, 'closeSource': 'official', 'closeTime': 1696017600257, 'companyName': 'Apple Inc', 'currency': 'USD', 'delayedPrice': 171.16, 'delayedPriceTime': 1696017588101, 'extendedChange': 0.12, 'extendedChangePercent': 0.0007, 'extendedPrice': 171.33, 'extendedPriceTime': 1696031994414, 'high': 173.07, 'highSource': '15 minute delayed price', 'highTime': 1696017599938, 'iexAskPrice': 0, 'iexAskSize': 0, 'iexBidPrice': 0, 'iexBidSize': 0, 'iexClose': 171.29, 'iexCloseTime': 1696017599245, 'iexLastUpdated': 1696017599245, 'iexMarketPercent': 0.01796985226860766, 'iexOpen': 172, 'iexOpenTime': 1695994200024, 'iexRealtimePrice': 171.29, 'iexRealtimeSize': 172, 'iexVolume': 931936, 'lastTradeTime': 1696017599938, 'latestPrice': 171.21, 'latestSource': 'Close', 'latestTime': 'September 29, 2023', 'l

## Parsing Our API Call

The API call that we executed in the last code block contains all of the information required to build our equal-weight S&P 500 strategy. 

With that said, the data isn't in a proper format yet. We need to parse it first.

In [25]:
price = data['latestPrice']
market_cap = data['marketCap']
print(price)
print(market_cap/1000000000000)

171.21
2.67673686072


## Adding Our Stocks Data to a Pandas DataFrame

The next thing we need to do is add our stock's price and market capitalization to a pandas DataFrame. Think of a DataFrame like the Python version of a spreadsheet. It stores tabular data.

In [26]:
my_columns = ['Ticker', 'Price', 'Change Percent', 'Market Capitalization', 'Number of Shares to Buy']
final_dataframe = pd.DataFrame(columns = my_columns)
final_dataframe

,Ticker,Price,Change Percent,Market Capitalization,Number of Shares to Buy


In [27]:
final_dataframe = final_dataframe.append(
                                        pd.Series(['AAPL',
                                                  data['latestPrice'],
                                                  data['changePercent'] * 100,
                                                  data['marketCap'],
                                                  'N/A'],
                                                 index = my_columns),
                                                 ignore_index = True)
final_dataframe

/tmp/ipykernel_27064/3395794865.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  final_dataframe = final_dataframe.append(


,Ticker,Price,Change Percent,Market Capitalization,Number of Shares to Buy
0,AAPL,171.21,0.305,2676736860720,N/A


## Looping Through The Tickers in Our List of Stocks

Using the same logic that we outlined above, we can pull data for all S&P 500 stocks and store their data in the DataFrame using a `for` loop.

In [28]:
final_dataframe = pd.DataFrame(columns = my_columns)

for stock in stocks['Ticker'][:5]:
    
    api_url = f'https://api.iex.cloud/v1/data/core/quote/{stock}?token={api_token}'
    print(api_url)
    data = requests.get(api_url).json()[0]
    
    series = pd.Series([data['symbol'], data['latestPrice'], np.multiply(data['changePercent'],100), data['marketCap'], 'N/A'], index = my_columns)
    
    final_dataframe = final_dataframe.append(series, ignore_index=True)

https://api.iex.cloud/v1/data/core/quote/A?token=pk_2232a975d7f0498bb545463065fb960b


/tmp/ipykernel_27064/106475867.py:11: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  final_dataframe = final_dataframe.append(series, ignore_index=True)


https://api.iex.cloud/v1/data/core/quote/AAL?token=pk_2232a975d7f0498bb545463065fb960b


/tmp/ipykernel_27064/106475867.py:11: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  final_dataframe = final_dataframe.append(series, ignore_index=True)


https://api.iex.cloud/v1/data/core/quote/AAP?token=pk_2232a975d7f0498bb545463065fb960b


/tmp/ipykernel_27064/106475867.py:11: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  final_dataframe = final_dataframe.append(series, ignore_index=True)


https://api.iex.cloud/v1/data/core/quote/AAPL?token=pk_2232a975d7f0498bb545463065fb960b


/tmp/ipykernel_27064/106475867.py:11: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  final_dataframe = final_dataframe.append(series, ignore_index=True)


https://api.iex.cloud/v1/data/core/quote/ABBV?token=pk_2232a975d7f0498bb545463065fb960b


/tmp/ipykernel_27064/106475867.py:11: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  final_dataframe = final_dataframe.append(series, ignore_index=True)


In [29]:
final_dataframe

,Ticker,Price,Change Percent,Market Capitalization,Number of Shares to Buy
0,A,111.82,-0.161,32717108867,N/A
1,AAL,12.81,-0.851,8369572805,N/A
2,AAP,55.93,1.562,3326171698,N/A
3,AAPL,171.21,0.305,2676736860720,N/A
4,ABBV,149.06,-2.095,263097858121,N/A


## Using Batch API Calls to Improve Performance

Batch API calls are one of the easiest ways to improve the performance of your code.

This is because HTTP requests are typically one of the slowest components of a script.

Also, API providers will often give you discounted rates for using batch API calls since they are easier for the API provider to respond to.

IEX Cloud limits their batch API calls to 100 tickers per request. Still, this reduces the number of API calls we'll make in this section from 500 to 5 - huge improvement! In this section, we'll split our list of stocks into groups of 100 and then make a batch API call for each group.

In [30]:
# Function sourced from 
# https://stackoverflow.com/questions/312443/how-do-you-split-a-list-into-evenly-sized-chunks
def chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

In [100]:
import warnings
import time

# To ignore all warnings
warnings.filterwarnings("ignore")

start_time = time.time()
symbol_groups = list(chunks(stocks['Ticker'], 505))

symbol_strings = []

for i in range(0, len(symbol_groups)):
    symbol_strings.append(','.join(symbol_groups[i]))
    
final_dataframe = pd.DataFrame(columns = my_columns)

for symbol_string in symbol_strings:
    batch_api_call_url = f'https://api.iex.cloud/v1/data/core/quote/{symbol_string}?token={api_token}'
    print(batch_api_call_url)
    data = requests.get(batch_api_call_url).json()
    
    for symbol in range(len(symbol_string.split(','))):
        
        series = pd.Series([data[symbol]['symbol'], data[symbol]['latestPrice'], data[symbol]['changePercent'], '{:.2f}B'.format(np.divide(data[symbol]['marketCap'], 1e9)) if data[symbol]['marketCap'] is not None else 'N/A', 'N/A'], index = my_columns)

        final_dataframe = final_dataframe.append(series, ignore_index=True)
        
end_time = time.time()
print(end_time - start_time)
final_dataframe

https://api.iex.cloud/v1/data/core/quote/A,AAL,AAP,AAPL,ABBV,ABC,ABMD,ABT,ACN,ADBE,ADI,ADM,ADP,ADSK,AEE,AEP,AES,AFL,AIG,AIV,AIZ,AJG,AKAM,ALB,ALGN,ALK,ALL,ALLE,ALXN,AMAT,AMCR,AMD,AME,AMGN,AMP,AMT,AMZN,ANET,ANSS,ANTM,AON,AOS,APA,APD,APH,APTV,ARE,ATO,ATVI,AVB,AVGO,AVY,AWK,AXP,AZO,BA,BAC,BAX,BBY,BDX,BEN,BF.B,BIIB,BIO,BK,BKNG,BKR,BLK,BLL,BMY,BR,BRK.B,BSX,BWA,BXP,C,CAG,CAH,CARR,CAT,CB,CBOE,CBRE,CCI,CCL,CDNS,CDW,CE,CERN,CF,CFG,CHD,CHRW,CHTR,CI,CINF,CL,CLX,CMA,CMCSA,CME,CMG,CMI,CMS,CNC,CNP,COF,COG,COO,COP,COST,COTY,CPB,CPRT,CRM,CSCO,CSX,CTAS,CTL,CTSH,CTVA,CTXS,CVS,CVX,CXO,D,DAL,DD,DE,DFS,DG,DGX,DHI,DHR,DIS,DISCA,DISCK,DISH,DLR,DLTR,DOV,DOW,DPZ,DRE,DRI,DTE,DUK,DVA,DVN,DXC,DXCM,EA,EBAY,ECL,ED,EFX,EIX,EL,EMN,EMR,EOG,EQIX,EQR,ES,ESS,ETFC,ETN,ETR,EVRG,EW,EXC,EXPD,EXPE,EXR,F,FANG,FAST,FB,FBHS,FCX,FDX,FE,FFIV,FIS,FISV,FITB,FLIR,FLS,FLT,FMC,FOX,FOXA,FRC,FRT,FTI,FTNT,FTV,GD,GE,GILD,GIS,GL,GLW,GM,GOOG,GOOGL,GPC,GPN,GPS,GRMN,GS,GWW,HAL,HAS,HBAN,HBI,HCA,HD,HES,HFC,HIG,HII,HLT,HOLX,HON,HPE,HPQ,HRB,HRL,HSIC

,Ticker,Price,Change Percent,Market Capitalization,Number of Shares to Buy
0,A,111.82,-0.00161,32.72B,N/A
1,AAL,12.81,-0.00851,8.37B,N/A
2,AAP,55.93,0.01562,3.33B,N/A
3,AAPL,171.21,0.00305,2676.74B,N/A
4,ABBV,149.06,-0.02095,263.10B,N/A
...,...,...,...,...,...
500,YUM,124.94,0.00693,35.01B,N/A
501,ZBH,112.22,-0.00213,23.45B,N/A
502,ZBRA,236.53,-0.00144,12.14B,N/A
503,ZION,34.89,0.02769,5.17B,N/A


## Calculating the Number of Shares to Buy

As you can see in the DataFrame above, we stil haven't calculated the number of shares of each stock to buy.

We'll do that next.

In [86]:
portfolio_size = ""

while not isinstance(portfolio_size, float):
    try:
        portfolio_size = input("Enter the value of your portfolio:")
        portfolio_size = float(portfolio_size)

    except ValueError:
        print("That's not a number! \nTry again:\n")


Enter the value of your portfolio: 1000000


In [88]:
position_size = float(portfolio_size)/len(final_dataframe.index)
print(f'For each stock: {position_size}$')

for i in range(0, len(final_dataframe['Ticker'])-1):
    final_dataframe.loc[i, 'Number of Shares to Buy'] = math.floor(position_size / final_dataframe['Price'][i])

final_dataframe

For each stock: 1980.1980198019803$


,Ticker,Price,Change Percent,Market Capitalization,Number of Shares to Buy
0,A,111.82,-0.00161,32717108867,17
1,AAL,12.81,-0.00851,8369572805,154
2,AAP,55.93,0.01562,3326171698,35
3,AAPL,171.21,0.00305,2676736860720,11
4,ABBV,149.06,-0.02095,263097858121,13
...,...,...,...,...,...
500,YUM,124.94,0.00693,35009597448,15
501,ZBH,112.22,-0.00213,23449955117,17
502,ZBRA,236.53,-0.00144,12143063237,8
503,ZION,34.89,0.02769,5168783690,56


## Formatting Our Excel Output

We will be using the XlsxWriter library for Python to create nicely-formatted Excel files.

XlsxWriter is an excellent package and offers tons of customization. However, the tradeoff for this is that the library can seem very complicated to new users. Accordingly, this section will be fairly long because I want to do a good job of explaining how XlsxWriter works.

### Initializing our XlsxWriter Object

In [114]:
writer = pd.ExcelWriter('recommended_trades.xlsx', engine='xlsxwriter')
final_dataframe.to_excel(writer,sheet_name = 'Recommended Trades', index=False)

### Creating the Formats We'll Need For Our `.xlsx` File

Formats include colors, fonts, and also symbols like `%` and `$`. We'll need four main formats for our Excel document:
* String format for tickers
* \\$XX.XX format for stock prices
* \\$XX,XXX format for market capitalization
* Integer format for the number of shares to purchase

In [115]:
background_color = '#0a0a23'
font_color = '#ffffff'

string_format = writer.book.add_format(
    {
        'font_color': font_color,
        'bg_color': background_color,
        'border': 1,
        'align': 'center'
    }
)

dollar_format = writer.book.add_format(
        {
            'num_format':'$0.00',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1,
            'align': 'center'
        }
    )

dollar_billons_format = writer.book.add_format(
        {
            'num_format':'$0.00, "B"',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1,
            'align': 'center'
        }
    )

integer_format = writer.book.add_format(
        {
            'num_format':'0',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1,
            'align': 'center'
        }
    )

percent_format = writer.book.add_format(
        {
            'num_format':'0.00%',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1,
            'align': 'center'
        }
    )

### Applying the Formats to the Columns of Our `.xlsx` File

We can use the `set_column` method applied to the `writer.sheets['Recommended Trades']` object to apply formats to specific columns of our spreadsheets.

Here's an example:

```python
writer.sheets['Recommended Trades'].set_column('B:B', #This tells the method to apply the format to column B
                     18, #This tells the method to apply a column width of 18 pixels
                     string_template #This applies the format 'string_template' to the column
                    )
```

In [116]:
# writer.sheets['Recommended Trades'].write('A1', 'Ticker', string_format)
# writer.sheets['Recommended Trades'].write('B1', 'Price', string_format)
# writer.sheets['Recommended Trades'].write('C1', 'Market Capitalization', string_format)
# writer.sheets['Recommended Trades'].write('D1', 'Number Of Shares to Buy', string_format)
# writer.sheets['Recommended Trades'].set_column('A:A', 20, string_format)
# writer.sheets['Recommended Trades'].set_column('B:B', 20, dollar_format)
# writer.sheets['Recommended Trades'].set_column('C:C', 20, dollar_format)
# writer.sheets['Recommended Trades'].set_column('D:D', 20, integer_format)

This code works, but it violates the software principle of "Don't Repeat Yourself". 

Let's simplify this by putting it in 2 loops:

In [117]:
columns_formats = {
    'A': ['Ticker', string_format],
    'B': ['Price', dollar_format],
    'C': ['Change Percent', percent_format],
    'D': ['Market Capitalization', dollar_format],
    'E': ['Number of Shares to Buy', integer_format]
}

print(columns_formats['A'][1])

for column in columns_formats.keys():
    writer.sheets['Recommended Trades'].set_column(f'{column}:{column}', 20, columns_formats[column][1])
    writer.sheets['Recommended Trades'].write(f'{column}1', columns_formats[column][0], string_format)


## Saving Our Excel Output

Saving our Excel file is very easy:

In [118]:
writer.save()